ដើម្បីរត់ notebook ទាំងអស់ដែលមាននៅខាងក្រោមនេះ ប្រសិនបើអ្នកមិនបានធ្វើរួចទេ អ្នកត្រូវតែដាក់កូដ key របស់ openai ក្នុងឯកសារ .env ជា `OPENAI_API_KEY`


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )

model = 'text-embedding-ada-002'

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"

បន្ទាប់មក យើងនឹងផ្ទុកសន្ទស្សន៍ Embedding ទៅក្នុង Dataframe របស់ Pandas។ សន្ទស្សន៍ Embedding ត្រូវបានរក្សាទុកក្នុងឯកសារ JSON ដែលមានឈ្មោះថា `embedding_index_3m.json`។ សន្ទស្សន៍ Embedding មានបណ្តុំនៃ Embeddings សម្រាប់សារាអត្ថបទ YouTube នៅរហូតដល់ចុងខែកក្កដា ២០២៣។


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

បន្ទាប់ យើងនឹងបង្កើតមុខងារ​មួយឈ្មោះ `get_videos` ដែលនឹងស្វែងរកក្នុង Embedding Index សម្រាប់សំណួរ។ មុខងារនឹងត្រឡប់មកវិញវីដេអូ ៥ ដ៏ខ្លាំងបំផុតដែលស្រដៀងនឹងសំណួរ។ មុខងារនេះដំណើរការដូចតទៅ៖

1. ជំហ៊ានទីមួយ គេចម្លង Embedding Index មួយ។
2. បន្ទាប់មក គណនាអាំងឌិចស៍សម្រាប់សំណួរដោយប្រើ OpenAI Embedding API។
3. បន្ទាប់មក បង្កើតជួរឈរថ្មីមួយក្នុង Embedding Index ដោយឈ្មោះ `similarity`។ ជួរឈរ `similarity` រួមមានភាពស្រដៀង cosine រវាង Embedding សម្រាប់សំណួរ និង Embedding សម្រាប់រាល់វិដេអូដែលមានផ្នែកដែលបានចែកចាយ។
4. បន្ទាប់មក ធ្វើការត្រង Embedding Index តាមជួរឈរ `similarity`។ Embedding Index ត្រូវបានត្រងត្រឹមតែរូបភាពដែលមានភាពស្រដៀង cosine ធំជាងឬស្មើ 0.75 ប៉ុណ្ណោះ។
5. ចុងក្រោយ Embedding Index ត្រូវបានច្រាសតាមជួរឈរ `similarity` ហើយត្រឡប់វិដេអូ ៥ ដ៏ខ្លាំងបំផុតត្រឡប់មកវិញ។


In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

មុខងារនេះសាមញ្ញបំផុត វាបង្ហាញតែលទ្ធផលនៃការស្វែងរកប៉ុណ្ណោះ។


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. ជាដំបូង, សន្ទស្សន៍ Embedding ត្រូវបានផ្ទុកចូលក្នុង Dataframe របស់ Pandas។
2. បន្ទាប់មក អ្នកប្រើត្រូវបានហៅឲ្យបញ្ចូលសំណួរ។
3. បន្ទាប់មក គោលការណ៍ `get_videos` ត្រូវបានហៅដើម្បីស្វែងរកសន្ទស្សន៍ Embedding សម្រាប់សំណួរនោះ។
4. ចុងក្រោយ គោលការណ៍ `display_results` ត្រូវបានហៅដើម្បីបង្ហាញលទ្ធផលទៅអ្នកប្រើ។
5. បន្ទាប់មក អ្នកប្រើត្រូវបានហៅឲ្យបញ្ចូលសំណួរផ្សេងទៀត។ ដំណើរការនេះទទួលបន្តដល់ពេលដែលអ្នកប្រើបញ្ចូល `exit`។

![](../../../../translated_images/km/notebook-search.1e320b9c7fcbb0bc.webp)

អ្នកនឹងត្រូវបានហៅឲ្យបញ្ចូលសំណួរ។ បញ្ចូលសំណួរមួយ ហើយចុច enter។ កម្មវិធីនឹងត្រឡប់មកវិញបញ្ជីវីដេអូដែលមានភាពពាក់ព័ន្ធនឹងសំនួរនោះ។ កម្មវិធីនឹងផ្តល់តំណទៅកាន់កន្លែងនៅក្នុងវីដេអូដែលចម្លើយសម្រាប់សំនួរនោះត្រូវបានស្ថិតនៅ។

ខាងក្រោមជាសំនួរដើម្បីសាកល្បង៖

- Azure Machine Learning គឺជាអ្វី?
- បញ្ច្រាសបណ្តាញប្រសាទ convolutional neural networks ដំណើរការយ៉ាងដូចម្តេច?
- បណ្តាញប្រសាទ neural network គឺជាអ្វី?
- ខ្ញុំអាចប្រើ Jupyter Notebooks ជាមួយ Azure Machine Learning បានទេ?
- ONNX គឺជាអ្វី?


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**បំណងជាក់លាក់**៖  
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំសម្រាប់ភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាបកប្រែដោយស្វ័យប្រវត្តិសាស្រ្តអាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាមូលដ្ឋានគួរត្រូវបានចាត់ទុកជាផ្លូវការជាអធិប្បាយ។ សម្រាប់ព័ត៌មានសំខាន់ ការបកប្រែដោយមនុស្សអ្នកជំនាញគឺត្រូវបានណែនាំ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ខុស ឬការបកចំនួនខុសដែលកើតមានពីការប្រើប្រាស់បកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
